# Frequentadores STAT — lake_prep_stat

Combina duas tabelas raw (ritmos de atualização diferentes) e salva
`frequentadores` em `lake_prep_stat`.

```
stat_P2 (Oracle ODBC)  →  stat_atendimento
    ↓  Dataflow Gen2 [diário]    → lake_prep_stat.raw_freq_corrente
    ↓  Dataflow Gen2 [anual]     → lake_prep_stat.raw_freq_historico
                                         ↓  Este notebook (Fabric)
                                   lake_prep_stat.frequentadores
```

---

### SQL — raw_freq_corrente (Dataflow Gen2, atualização **diária**)

Últimas 52 semanas completas (ISO, segunda a domingo) + semana em curso.

```sql
SELECT *
FROM stat_atendimento
WHERE TRUNC(stat_data) >= TRUNC(SYSDATE, 'IW') - 364
```

> `TRUNC(SYSDATE, 'IW')` = segunda-feira da semana atual.  
> `- 364` = 52 semanas × 7 dias atrás, ou seja, início da semana há exatamente 52 semanas.

---

### SQL — raw_freq_historico (Dataflow Gen2, atualização **anual**)

De 2009 até o último dia do ano anterior ao corrente.

```sql
SELECT *
FROM stat_atendimento
WHERE TRUNC(stat_data) >= TO_DATE('01/01/2009', 'DD/MM/YYYY')
  AND TRUNC(stat_data) <  TRUNC(SYSDATE, 'YYYY')
```

> `TRUNC(SYSDATE, 'YYYY')` = 01/01 do ano corrente, portanto `<` exclui o ano atual
> e inclui até 31/12 do ano anterior.

---

**Sobreposição esperada:** corrente e histórico se sobrepõem nos meses do ano anterior
que caem dentro das últimas 52 semanas. O notebook elimina duplicatas no UNION.

In [ ]:
import pandas as pd
import numpy as np


def read_raw(table_name: str) -> pd.DataFrame:
    return spark.sql(f'SELECT * FROM lake_prep_stat.dbo.{table_name}').toPandas()


def save_prep(df: pd.DataFrame, table_name: str) -> None:
    spark.createDataFrame(df).write.mode('overwrite').saveAsTable(f'lake_prep_stat.{table_name}')

## 1. Carregar e combinar as tabelas raw

In [ ]:
def _normaliza_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [col.split('.')[-1].lower() for col in df.columns]
    return df


corrente_df  = _normaliza_cols(read_raw('raw_freq_corrente'))
historico_df = _normaliza_cols(read_raw('raw_freq_historico'))

print(f'raw_freq_corrente:  {corrente_df.shape}')
print(f'raw_freq_historico: {historico_df.shape}')

# UNION — dedup em (uor_codigo, stat_data): uma linha por unidade por dia
# Histórico carregado primeiro; corrente sobrescreve registros duplicados na zona de sobreposição
raw_freq_df = (
    pd.concat([historico_df, corrente_df], ignore_index=True)
    .drop_duplicates(subset=['uor_codigo', 'stat_data'], keep='last')
)

print(f'após UNION + dedup:  {raw_freq_df.shape}')

## 2. Transformações

In [ ]:
def transform_frequentadores(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Tipos base
    df['stat_data']  = pd.to_datetime(df['stat_data'], errors='coerce')
    df['uor_codigo'] = pd.to_numeric(df['uor_codigo'], errors='coerce').astype('Int64')

    # atend_qtde_pessoas → frequentadores
    df = df.rename(columns={'atend_qtde_pessoas': 'frequentadores'})
    df['frequentadores'] = pd.to_numeric(df['frequentadores'], errors='coerce').fillna(0).astype(int)

    # Periodo — marcos do período COVID
    # 1º = antes do fechamento (≤ 16/03/2020)
    # 2º = fechamento/reabertura gradual (≤ 23/08/2020)
    # 3º = pós-reabertura
    df['periodo'] = np.select(
        [
            df['stat_data'] <= pd.Timestamp('2020-03-16 23:59:59'),
            df['stat_data'] <= pd.Timestamp('2020-08-23 23:59:59'),
        ],
        ['1º', '2º'],
        default='3º',
    )

    # Condições compartilhadas por freq_blocos e freq_blocos_ordem
    f = df['frequentadores']
    conds = [
        f == 0,
        f < 1000,
        f < 2000,
        f < 3000,
        f < 4000,
        f < 5000,
        f < 10000,
        f < 15000,
        f < 20000,
    ]

    df['freq_blocos'] = np.select(
        conds,
        ['0', '< 1000', '1 mil a 1.999', '2 mil a 2.999', '3 mil a 3.999',
         '4 mil a 4.999', '5 mil a 9.999', '10 mil a 14.999', '15 mil a 19.999'],
        default='20.000 ou +',
    )

    df['freq_blocos_ordem'] = np.select(
        conds,
        [0, 1, 2, 3, 4, 5, 6, 7, 8],
        default=9,
    ).astype(int)

    # link = str(uor_codigo) + data no formato DD/MM/YYYY (sem hora)
    # Equivale a Text.Combine + Text.BeforeDelimiter(_, " ") do Power Query
    df['link'] = (
        df['uor_codigo'].astype(str) + df['stat_data'].dt.strftime('%d/%m/%Y')
    )

    # Reordena: colunas originais primeiro, depois as derivadas
    derivadas = ['periodo', 'frequentadores', 'freq_blocos', 'freq_blocos_ordem', 'link']
    originais = [c for c in df.columns if c not in derivadas]
    df = df[originais + derivadas]

    return df.sort_values('stat_data', ascending=False).reset_index(drop=True)

In [ ]:
frequentadores_df = transform_frequentadores(raw_freq_df)
print(f'frequentadores_df: {frequentadores_df.shape}')
print()
print('periodo:')
print(frequentadores_df['periodo'].value_counts())
print()
print('freq_blocos:')
print(frequentadores_df['freq_blocos'].value_counts().reindex(
    ['0', '< 1000', '1 mil a 1.999', '2 mil a 2.999', '3 mil a 3.999',
     '4 mil a 4.999', '5 mil a 9.999', '10 mil a 14.999', '15 mil a 19.999', '20.000 ou +']
))
print()
print('Exemplo de link:')
print(frequentadores_df['link'].head(3).tolist())

## 3. Salvar em lake_prep_stat

In [ ]:
save_prep(frequentadores_df, 'frequentadores')
print('Salvo em lake_prep_stat.frequentadores')